# 01 - The Self-Improving Copilot

The CityOps copilot you built in the prerequisite lab could *advise*. In this notebook it learns to *act* - and to get better at acting:

1. **Tool registry** - its capabilities become rows it retrieves by meaning
2. **Agent loop** - the model calls tools; every call is captured as a full trajectory
3. **Verified success** - an LLM judge audits each run against the database record; only verified runs count (no more `success = bool(final)`) 
4. **Workflow dedup** - recurring intents merge via an embedding gray band + LLM confirmation
5. **Skill harvest & lifecycle** - what recurs *and verifiably works* is distilled into a `SKILL.md`, born **provisional**, and earns **approved** (or **retired**) through linked outcomes

Each mechanism closes a gap documented in `memory_promotion_review.md`: promotion without curation, success without verification, growth without forgetting.

## 0 · Setup

Connect, verify notebook 00's artifacts, and stand up the in-database embedder.

In [ ]:
import array
import json
import uuid

from cityops_harness.checks import ok, check
from cityops_harness.config import load_settings
from cityops_harness.db import get_connection
from cityops_harness.llm import chat_model
from cityops_harness.state import require, verify
from cityops_harness.tracing import init_tracing
from cityops_harness import improve

settings = load_settings()
conn = get_connection(settings)
require(conn, "00_setup")
CHAT = chat_model(settings)
HANDLER = init_tracing(settings)
ok(f"connected - provider={settings.llm_provider}, langfuse={settings.langfuse_mode}")

In [ ]:
# The same IEmbedder bridge the CityOps lab uses: embeddings are computed
# inside the database via VECTOR_EMBEDDING() - no text egress, no API cost.
import asyncio
import numpy as np
from langchain_oracledb.embeddings.oracleai import OracleEmbeddings
from oracleagentmemory.apis.embedders.embedder import IEmbedder


class OracleONNXEmbedder(IEmbedder):
    def __init__(self, conn, model_name):
        self.provider = OracleEmbeddings(
            conn=conn, params={"provider": "database", "model": model_name}
        )

    def embed(self, texts, *, is_query=False):
        return np.asarray(self.provider.embed_documents(list(texts)), dtype=np.float32)

    async def embed_async(self, texts, *, is_query=False):
        return await asyncio.to_thread(self.embed, texts, is_query=is_query)


embedder = OracleONNXEmbedder(conn, settings.embed_model)
check(embedder.embed(["hello"]).shape == (1, 384), "in-database embedder returns 384-dim vectors")

## 1 · Domain bootstrap (pre-built)

The system of record from the CityOps lab, loaded idempotently: the asset registry and
~inspection findings with in-database embeddings. Re-running this section is safe.

In [ ]:
from pathlib import Path

_data_dir = Path("data") if Path("data").exists() else Path("../data")
with open(_data_dir / "maintenance_logs.json") as f:
    _logs = json.load(f)
with open(_data_dir / "inspection_reports.json") as f:
    _reports = json.load(f)
_asset_names = sorted({x["asset_name"] for x in _logs} | {x["asset_name"] for x in _reports})


def classify_asset(name: str) -> str:
    n = name.lower()
    for key, cls in (("bridge", "bridge"), ("tower", "tower"), ("pump", "pump_station"),
                     ("tunnel", "tunnel"), ("plant", "plant"), ("station", "station")):
        if key in n:
            return cls
    return "structure"


def _table_exists(name: str) -> bool:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM user_tables WHERE table_name = :t", t=name)
        return cur.fetchone()[0] > 0


if not _table_exists("CITY_ASSET"):
    with conn.cursor() as cur:
        cur.execute("""CREATE TABLE CITY_ASSET (
            asset_id     VARCHAR2(128) PRIMARY KEY,
            asset_class  VARCHAR2(32) NOT NULL,
            created_at   TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        cur.executemany("INSERT INTO CITY_ASSET (asset_id, asset_class) VALUES (:1, :2)",
                        [(a, classify_asset(a)) for a in _asset_names])
    conn.commit()


def get_asset(asset_id: str):
    with conn.cursor() as cur:
        cur.execute("SELECT asset_id, asset_class FROM CITY_ASSET WHERE asset_id = :id", id=asset_id)
        row = cur.fetchone()
    return {"asset_id": row[0], "asset_class": row[1]} if row else None

ok(f"{len(_asset_names)} assets registered")

In [ ]:
import os

DEMO_FINDING_COUNT = int(os.getenv("DEMO_FINDING_COUNT", "60"))

if not _table_exists("CITY_INSPECTION_FINDING"):
    with conn.cursor() as cur:
        cur.execute("""CREATE TABLE CITY_INSPECTION_FINDING (
            finding_id      VARCHAR2(64) PRIMARY KEY,
            asset_id        VARCHAR2(128) NOT NULL,
            inspector       VARCHAR2(128),
            overall_grade   VARCHAR2(2),
            category        VARCHAR2(32),
            severity        VARCHAR2(16),
            description     CLOB NOT NULL,
            recommendation  CLOB,
            days_ago        NUMBER,
            embedding       VECTOR(384) NOT NULL,
            created_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            CONSTRAINT fk_finding_asset FOREIGN KEY (asset_id) REFERENCES CITY_ASSET(asset_id))""")
        try:
            cur.execute("""CREATE VECTOR INDEX city_finding_embedding_idx
                ON CITY_INSPECTION_FINDING (embedding)
                ORGANIZATION INMEMORY NEIGHBOR GRAPH DISTANCE COSINE
                WITH TARGET ACCURACY 95 PARAMETERS (TYPE HNSW, M 16, EFCONSTRUCTION 200)""")
        except Exception as e:
            print(f"  (skipped HNSW index: {e})")
    conn.commit()

with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM CITY_INSPECTION_FINDING")
    _existing = cur.fetchone()[0]

if _existing == 0:
    rows = []
    for report in _reports:
        if len(rows) >= DEMO_FINDING_COUNT:
            break
        for finding in report["findings"]:
            if len(rows) >= DEMO_FINDING_COUNT:
                break
            vec = array.array('f', embedder.embed([finding["description"]])[0].tolist())
            rows.append({
                "finding_id": str(uuid.uuid4())[:12], "asset_id": report["asset_name"],
                "inspector": report["inspector"], "overall_grade": report["overall_grade"],
                "category": finding["category"], "severity": finding["severity"],
                "description": finding["description"], "recommendation": finding["recommendation"],
                "days_ago": report["days_ago"], "embedding": vec,
            })
    with conn.cursor() as cur:
        cur.executemany("""INSERT INTO CITY_INSPECTION_FINDING
            (finding_id, asset_id, inspector, overall_grade, category, severity,
             description, recommendation, days_ago, embedding)
            VALUES (:finding_id, :asset_id, :inspector, :overall_grade, :category, :severity,
                    :description, :recommendation, :days_ago, :embedding)""", rows)
    conn.commit()
    _existing = len(rows)
ok(f"{_existing} findings in the system of record")

In [ ]:
def log_finding(asset_id, inspector, category, severity, description,
                recommendation="", overall_grade=None, days_ago=0):
    finding_id = str(uuid.uuid4())[:12]
    vec = array.array('f', embedder.embed([description])[0].tolist())
    with conn.cursor() as cur:
        cur.execute("""INSERT INTO CITY_INSPECTION_FINDING
             (finding_id, asset_id, inspector, overall_grade, category, severity,
              description, recommendation, days_ago, embedding)
           VALUES (:finding_id, :asset_id, :inspector, :overall_grade, :category, :severity,
                   :description, :recommendation, :days_ago, :embedding)""",
            finding_id=finding_id, asset_id=asset_id, inspector=inspector,
            overall_grade=overall_grade, category=category, severity=severity,
            description=description, recommendation=recommendation,
            days_ago=days_ago, embedding=vec)
    conn.commit()
    return finding_id


def find_similar_findings(description, asset_id=None, category=None, k=3):
    query_vec = array.array('f', embedder.embed([description])[0].tolist())
    sql = """SELECT finding_id, asset_id, inspector, overall_grade, category, severity,
                    description, recommendation, days_ago,
                    VECTOR_DISTANCE(embedding, :q, COSINE) AS score
               FROM CITY_INSPECTION_FINDING
              WHERE (:asset_id IS NULL OR asset_id = :asset_id)
                AND (:category IS NULL OR category = :category)
              ORDER BY score FETCH FIRST :k ROWS ONLY"""
    with conn.cursor() as cur:
        cur.execute(sql, q=query_vec, asset_id=asset_id, category=category, k=k)
        cols = [d[0].lower() for d in cur.description]
        out = []
        for r in cur.fetchall():
            row = dict(zip(cols, r))
            for key in ("description", "recommendation"):
                v = row.get(key)
                if v is not None and hasattr(v, "read"):
                    row[key] = v.read()
            out.append(row)
    return out

ASSET_A = "Harbor Bridge" if "Harbor Bridge" in _asset_names else _asset_names[0]
ASSET_B = next(a for a in _asset_names if a != ASSET_A)
ok(f"domain tools ready - demo assets: {ASSET_A!r}, {ASSET_B!r}")

## 2 · The harness registries

Three plain tables the harness *writes to as it learns*:

- `HARNESS_TOOLS` - capabilities, retrieved by meaning
- `HARNESS_WORKFLOW` - captured trajectories with **verified** outcome counts
- `HARNESS_SKILLS` - distilled `SKILL.md` documents with a **lifecycle** (provisional → approved | retired), linked back to outcomes via `skill_id`

In [ ]:
for ddl in [
    """CREATE TABLE HARNESS_TOOLS (
        name           VARCHAR2(64) PRIMARY KEY,
        description    VARCHAR2(1000) NOT NULL,
        params_schema  CLOB,
        embedding      VECTOR(384) NOT NULL,
        created_at     TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""",
    """CREATE TABLE HARNESS_WORKFLOW (
        workflow_id        VARCHAR2(64) PRIMARY KEY,
        intent             VARCHAR2(2000) NOT NULL,
        intent_embedding   VECTOR(384) NOT NULL,
        steps              CLOB NOT NULL,
        tools_used         VARCHAR2(1000),
        occurrences        NUMBER DEFAULT 1,
        verified_successes NUMBER DEFAULT 0,
        failures           NUMBER DEFAULT 0,
        promoted           VARCHAR2(1) DEFAULT 'N',
        last_seen          TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        created_at         TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""",
    """CREATE TABLE HARNESS_SKILLS (
        skill_id           VARCHAR2(64) PRIMARY KEY,
        name               VARCHAR2(128) NOT NULL,
        description        VARCHAR2(1000) NOT NULL,
        content            CLOB NOT NULL,
        sha                VARCHAR2(64) NOT NULL,
        status             VARCHAR2(16) DEFAULT 'provisional'
                           CHECK (status IN ('provisional','approved','retired')),
        schema_sha         VARCHAR2(64),
        source_workflow_id VARCHAR2(64),
        embedding          VECTOR(384) NOT NULL,
        uses               NUMBER DEFAULT 0,
        verified_successes NUMBER DEFAULT 0,
        failures           NUMBER DEFAULT 0,
        created_at         TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        updated_at         TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""",
]:
    table = ddl.split("CREATE TABLE ")[1].split(" ")[0].strip()
    if not _table_exists(table):
        with conn.cursor() as cur:
            cur.execute(ddl)
        conn.commit()
ok("HARNESS_TOOLS / HARNESS_WORKFLOW / HARNESS_SKILLS ready")

In [ ]:
def current_schema_sha():
    """Fingerprint of the domain schema a skill was distilled against.
    Skills whose stored schema_sha no longer matches are flagged stale at retrieval."""
    with conn.cursor() as cur:
        cur.execute("""SELECT table_name, column_name, data_type
                         FROM user_tab_columns
                        WHERE table_name LIKE 'CITY!_%' ESCAPE '!'
                           OR table_name LIKE 'HARNESS!_%' ESCAPE '!'""")
        cols = cur.fetchall()
    return improve.compute_schema_sha(cols)

SCHEMA_SHA = current_schema_sha()
ok(f"schema fingerprint {SCHEMA_SHA[:12]}...")